In [1]:
from experiment import Experiment
from database import PatchDataset
from models import TokenEncoder, MultiEnsembleEncoder
from metrics import BalancedMAE
from optimizer import AdamOptimizer
import pandas as pd
from utils import analyze_model

epochs = [55, 51, 51, 51]
variables = ['t2m', 't2m_clim', 'sd', 'sd_clim', 'tp', 'tp_clim', 'sst', 'sst_clim', 'snow_cover', 'lsm', 'z', 'sdor']
global_variables = ['t2m', 'sd', 'tp', 'sst', 'snow_cover']
inm_variables = ['t2m', 'swe', 'olr', 'hlt', 'mslp', 'tp', 'h500', 'snow_cover', 'u850', 'v850',  'ts', 'ww', 't2m_clim',
                 'ts_clim', 'swe_clim', 'tp_clim']

In [2]:
from matplotlib import pyplot as plt
from scales import normalize, denormalize

def model_samples(ds, model, experiment_name, submodel=None, prefix=''):
    model = model.to('cpu')
    sample = 0
    for x, y in ds:
        sample += 1
        inputs = {key: value.to('cpu') for key, value in x.items()}
        targets = {key: value.to('cpu') for key, value in y.items()}
        outs = model(inputs)
        if submodel is not None:
            targets = submodel(targets)
        for k in outs:
            for b in range(1):
                if submodel is None:
                    pred_data = denormalize(outs[k][b, 0, 0, :, :], k).detach().numpy()
                    target_data = targets[prefix + k][b, 0, :, :].detach().numpy()
                else:
                    ch = 0 if outs[k].shape[1] == 1 else 2
                    pred_data = outs[k][b, ch, 0, :, :].detach().numpy()
                    target_data = targets[prefix + k][b, ch, 0, :, :].detach().numpy()
                vmin = target_data.min()
                vmax = target_data.max()
                fig, axes = plt.subplots(1, 2, figsize=(12, 5))
                im1 = axes[0].imshow(pred_data, cmap='bwr', vmin=vmin, vmax=vmax, origin='lower')
                axes[0].set_title('Prediction')
                axes[0].axis('off')
                im2 = axes[1].imshow(target_data, cmap='bwr', vmin=vmin, vmax=vmax, origin='lower')
                axes[1].set_title('Target')
                axes[1].axis('off')
                fig.colorbar(im1, ax=axes)
                plt.savefig(f'../experiments/{experiment_name}/images/{k}_{b+sample}.png', dpi=100, bbox_inches='tight')
                plt.close()
        if sample == 10:
            break

In [3]:
ds = PatchDataset(
    input_variables=['inm_' + v for v in inm_variables],
    target_variables=['inm_' + v for v in inm_variables],
    modes={
        'train': {"t_min": '19910101', 't_max': '20201231', 'epoch_size': 2500, 'batch_size': 1},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 500, 'batch_size': 1},
    },
    era_scales=[],
    inm_scales=[{'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 32, 'xyStep': 1, 'tStep': 1}],
    target_scale={'xSize': 16, 'ySize': 16, 'tSize': 32, 'xyStep': 1, 'tStep': 1},
    num_workers=8,
    mask='snow'
)
variable_encoders={var: f'../experiments/ensemble_encoder_{var}' for var in inm_variables if '_clim' not in var and var != 'snow_cover'}
model = TokenEncoder(in_channels=4,
            variables=inm_variables,
            scale='inm_regional',
            variable_encoders=variable_encoders,
            token_dim=64,
            first_conv_dim=32,
            spatial_blocks=3,
            temporal_blocks=2,
            n_convs_per_block=0,
            kernel_size=3,
            cond_dim=16)
metric = BalancedMAE(inm_variables[0], 'inm_', MultiEnsembleEncoder(variable_encoders), alpha=0.995)
optimizer = AdamOptimizer(lr=0.0005)
experiment = Experiment(f'inm_encoder', ds, model, metric, optimizer, modes=['train', 'test'])
experiment.run(epochs[0], save_every_n_epochs=1)
ds.set_mode('test')
ds.set_seed(0)
model_samples(ds, experiment.model, 'inm_encoder', prefix='inm_', submodel=MultiEnsembleEncoder(variable_encoders))

In [4]:
ds = PatchDataset(
    input_variables=variables,
    target_variables=variables,
    modes={
        'train': {"t_min": '19810101', 't_max': '20201231', 'epoch_size': 2500, 'batch_size': 4},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 500, 'batch_size': 4},
    },
    era_scales=[{'id': 'local', 'xSize': 64, 'ySize': 64, 'tSize': 1, 'xyStep': 1, 'tStep': 8}],
    inm_scales=[],
    target_scale={'xSize': 64, 'ySize': 64, 'tSize': 1, 'xyStep': 1, 'tStep': 8},
    num_workers=8,
    raw_era_target=True,
    mask='snow'
)
model = TokenEncoder(in_channels=1,
            variables=variables,
            scale='local',
            variable_encoders={},
            token_dim=64,
            first_conv_dim=16,
            spatial_blocks=5,
            temporal_blocks=0,
            n_convs_per_block=0,
            kernel_size=3,
            cond_dim=16)
metric = BalancedMAE('t2m', '', MultiEnsembleEncoder({}), alpha=0.995)
optimizer = AdamOptimizer(lr=0.0005)
experiment = Experiment(f'era_local_encoder', ds, model, metric, optimizer, modes=['train', 'test'])
experiment.run(epochs[1], save_every_n_epochs=1)
ds.set_mode('test')
ds.set_seed(0)
model_samples(ds, experiment.model, 'era_local_encoder')

train epoch  51:   0%|          | 0/2500 [00:00<?, ?batch/s]

test  epoch  51:   0%|          | 0/500 [00:00<?, ?batch/s]

In [5]:
ds = PatchDataset(
    input_variables=variables,
    target_variables=variables,
    modes={
        'train': {"t_min": '19810101', 't_max': '20201231', 'epoch_size': 2500, 'batch_size': 4},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 500, 'batch_size': 4},
    },
    era_scales=[{'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 8, 'xyStep': 4, 'tStep': 8}],
    inm_scales=[],
    target_scale={'xSize': 16, 'ySize': 16, 'tSize': 8, 'xyStep': 4, 'tStep': 8},
    num_workers=8,
    raw_era_target=True,
    mask='snow'
)
model = TokenEncoder(in_channels=1,
            variables=variables,
            scale='regional',
            variable_encoders={},
            token_dim=64,
            first_conv_dim=16,
            spatial_blocks=4,
            temporal_blocks=0,
            n_convs_per_block=0,
            kernel_size=3,
            cond_dim=16)
metric = BalancedMAE('t2m', '', MultiEnsembleEncoder({}), alpha=0.995)
optimizer = AdamOptimizer(lr=0.0005)
experiment = Experiment(f'era_regional_encoder', ds, model, metric, optimizer, modes=['train', 'test'])
experiment.run(epochs[2], save_every_n_epochs=1)
ds.set_mode('test')
ds.set_seed(0)
model_samples(ds, experiment.model, 'era_regional_encoder')

train epoch  51:   0%|          | 0/2500 [00:00<?, ?batch/s]

test  epoch  51:   0%|          | 0/500 [00:00<?, ?batch/s]

In [6]:
ds = PatchDataset(
    input_variables=global_variables,
    target_variables=global_variables,
    modes={
        'train': {"t_min": '19810101', 't_max': '20201231', 'epoch_size': 2500, 'batch_size': 1},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 500, 'batch_size': 1},
    },
    era_scales=[{'id': 'global', 'xSize': 90, 'ySize': 22, 'tSize': 6, 'xyStep': 16, 'tStep': 32, 'fixY': 4}],
    inm_scales=[],
    target_scale={'xSize': 90, 'ySize': 22, 'tSize': 6, 'xyStep': 16, 'tStep': 32, 'fixY': 4},
    num_workers=8,
    raw_era_target=True,
    mask='snow'
)
model = TokenEncoder(in_channels=1,
            variables=global_variables,
            scale='global',
            variable_encoders={},
            token_dim=64,
            first_conv_dim=16,
            spatial_blocks=5,
            temporal_blocks=0,
            n_convs_per_block=0,
            kernel_size=3,
            cond_dim=16,
            paddings=[(1, 0), ((2, 1), (1, 0)), ((1, 2), (0, 1)), (2, 1)])
metric = BalancedMAE('t2m', '', MultiEnsembleEncoder({}), alpha=0.995)
optimizer = AdamOptimizer(lr=0.0005)
experiment = Experiment(f'era_global_encoder', ds, model, metric, optimizer, modes=['train', 'test'])
experiment.run(epochs[3], save_every_n_epochs=1)
ds.set_mode('test')
ds.set_seed(0)
model_samples(ds, experiment.model, 'era_global_encoder')

train epoch  51:   0%|          | 0/2500 [00:00<?, ?batch/s]

test  epoch  51:   0%|          | 0/500 [00:00<?, ?batch/s]

In [7]:
#df1 = pd.read_csv('../experiments/era_local_encoder_without_film/loss_history.csv').rename(columns={'train': 'train0', 'test': 'test0'})
#df2 = pd.read_csv('../experiments/era_local_encoder_with_film/loss_history.csv').rename(columns={'train': 'train1', 'test': 'test1'})
#df3 = pd.read_csv('../experiments/era_local_encoder_with_film_large_lr/loss_history.csv').rename(columns={'train': 'train2', 'test': 'test2'})
#df = df1.merge(df2, on='epoch', how='outer')
#df = df.merge(df3, on='epoch', how='outer')

In [8]:
#import numpy as np
#print(np.argmax(df['train1'] < np.min(df['train2'])) - len(df3))
#plt.plot(df['train0'])
#plt.plot(df['train1'])
#plt.plot(df['train2'])
#plt.plot(df['epoch'], df['train1']*0 + np.min(df['train1']))
#plt.ylim([0.01, 0.04])
#plt.show()
#plt.plot(df['test0'])
#plt.plot(df['test1'])
#plt.plot(df['test2'])
#plt.plot(df['epoch'], df['test1']*0 + np.min(df['test1']))
#plt.ylim([0.01, 0.04])
#plt.show()

In [1]:
from tqdm.auto import tqdm
import torch
from experiment import Experiment
from database import PatchDataset
import os

additional_variables = ['cos_lat', 'cos_lon', 'sin_lon', 'sin_lon', 'cos_day', 'sin_day', 'year_norm', 'lead_time',
              'inm_cos_lat', 'inm_cos_lon', 'inm_sin_lon', 'inm_cos_day', 'inm_sin_day', 'inm_year_norm', 'inm_lead_time']
variables = ['t2m', 't2m_clim', 'sd', 'sd_clim', 'tp', 'tp_clim', 'sst', 'sst_clim', 'snow_cover', 'lsm', 'z', 'sdor']
inm_variables = ['t2m', 'swe', 'olr', 'hlt', 'mslp', 'tp', 'h500', 'snow_cover', 'u850', 'v850',  'ts', 'ww', 't2m_clim',
                 'ts_clim', 'swe_clim', 'tp_clim']

epochs = 10
additional_variables_set = set(additional_variables)

In [2]:
ds = PatchDataset(
    input_variables=['inm_' + v for v in inm_variables] + variables + additional_variables,
    target_variables=['sd', 'sd_clim', 't2m', 't2m_clim', 'snow_cover', 'lat', 'lon', 'day', 'year', 'inm_lead_time'],
    modes={
        'train': {"t_min": '19910101', 't_max': '20201231', 'epoch_size': 1000, 'batch_size': 4},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 200, 'batch_size': 4},
    },
    era_scales=[{'id': 'local', 'xSize': 64, 'ySize': 64, 'tSize': 1, 'xyStep': 1, 'tStep': 8},
                {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 8, 'xyStep': 4, 'tStep': 8},
                {'id': 'global', 'xSize': 90, 'ySize': 22, 'tSize': 6, 'xyStep': 16, 'tStep': 32, 'fixY': 4}],
    inm_scales=[{'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 32, 'xyStep': 1, 'tStep': 1}],
    target_scale={'xSize': 16, 'ySize': 16, 'tSize': 32, 'xyStep': 4, 'tStep': 1},
    num_workers=8,
    mask='snow'
)

experiments = ['inm_encoder', 'era_local_encoder', 'era_regional_encoder', 'era_global_encoder']
models = [Experiment(name, checkpoint='e050').model for name in experiments]
for i in range(len(models)):
    models[i].eval()

In [3]:
for epoch in range(0, epochs+2):
    path = '../data/prepared_tokens/test.pt' if epoch == 0 else f'../data/prepared_tokens/train{epoch:03d}.pt'
    if epoch == epochs + 1:
        path = '../data/prepared_tokens/fulltest.pt'
    if os.path.exists(path):
        continue

    epoch_data = []
    if epoch <= epochs:
        ds.set_mode('test' if epoch == 0 else 'train')
        ds.set_seed(epoch)
    else:
        ds.set_full('test')

    with torch.no_grad():
        for x, y in tqdm(ds.loader, desc=f'Epoch {epoch}'):
            batch = {
                key: value
                for key, value in x.items()
                if any(key.startswith(var) for var in additional_variables_set)
            }
            batch['y'] = dict(y)

            for i in range(len(models)):
                tokens, shape, variables = models[i].encode(x)
                batch[f'tokens_{experiments[i]}'] = tokens
                batch[f'shape_{experiments[i]}'] = shape
                batch[f'variables_{experiments[i]}'] = variables

            epoch_data.append(batch)

    torch.save(epoch_data, path, pickle_protocol=5)

Epoch 0:   0%|          | 0/200 [00:00<?, ?it/s]

In [4]:
from tqdm.auto import tqdm
import torch
from experiment import Experiment
from database import PatchDataset
import os
import numpy as np
from utils import to_device

base_path = '../data/prepared_tokens2'
scales=[{'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 4, 'xyStep': 1, 'tStep': 1},
        {'id': 'local', 'xSize': 64, 'ySize': 64, 'tSize': 1, 'xyStep': 1, 'tStep': 8},
        {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 8, 'xyStep': 4, 'tStep': 8},
        {'id': 'global', 'xSize': 90, 'ySize': 22, 'tSize': 6, 'xyStep': 16, 'tStep': 32, 'fixY': 4}]

variables = ['t2m', 't2m_clim', 'sd', 'sd_clim', 'tp', 'tp_clim', 'sst', 'sst_clim', 'snow_cover', 'lsm', 'z', 'sdor']
inm_variables = ['t2m', 'swe', 'olr', 'hlt', 'mslp', 'tp', 'h500', 'snow_cover', 'u850', 'v850',  'ts', 'ww', 't2m_clim',
                 'ts_clim', 'swe_clim', 'tp_clim']
inm_variables = ['inm_' + v for v in inm_variables]
tasks = []
for v in inm_variables + variables:
    if 'inm_' in v:
        tasks.append((v, 0))
    else:
        tasks.append((v, 1))
        tasks.append((v, 2))
        if 'clim' not in v and v != 'lsm' and v != 'z' and v != 'sdor':
            tasks.append((v, 3))

In [6]:
experiments = ['inm_encoder', 'era_local_encoder', 'era_regional_encoder', 'era_global_encoder']
models = [Experiment(name, checkpoint='e050').model for name in experiments]
for i in range(len(models)):
    models[i].eval()

for v, i in tasks:
    print(v)
    if os.path.exists(f"{base_path}/{v}_{scales[i]['id']}.npy"):
        continue

    ds = PatchDataset(
        input_variables=[v],
        target_variables=[],
        modes={
            'train': {"t_min": '19910101', 't_max': '20201231', 'epoch_size': 1, 'batch_size': 2**8},
        },
        era_scales=[] if i == 0 else [scales[i]],
        inm_scales=[] if i > 0 else [scales[i]],
        target_scale={'xSize': 16, 'ySize': 16, 'tSize': 4, 'xyStep': 4, 'tStep': 1},
        num_workers=8
    )

    ds.set_full('train')
    if not os.path.exists(f'{base_path}/xytl.pt'):
        xytl = [(x, y, t, l) for (x, y, t), l in zip(ds.xyt, ds.lead_times)]
        torch.save(xytl, f'{base_path}/xytl.pt')

    encoded_map = {}
    data = []
    idx = []
    with torch.no_grad():
        pbar = tqdm(ds.loader, unit='batch', mininterval=0.5)
        for x, y in pbar:
            inputs = {key: to_device(value, next(models[i].parameters()).device) for key, value in x.items()}
            out, _, _ = models[i].encode(inputs)
            out = out.cpu().numpy().astype(np.float16)
            for j in range(out.shape[0]):
                tokens = np.ascontiguousarray(out[j].astype('<f2'))
                key = tokens.tobytes()
                if key not in encoded_map:
                    encoded_map[key] = len(data)
                    idx.append(len(data))
                    data.append(tokens)
                else:
                    idx.append(encoded_map[key])
            pbar.set_description(f'{v}_{i}: {len(idx)}/{len(data)}')
    data = np.concatenate(data, axis=0)
    idx = np.array(idx).astype(np.int32)
    np.save(f"{base_path}/{v}_{scales[i]['id']}.npy", data)
    np.save(f"{base_path}/{v}_{scales[i]['id']}_idx.npy", idx)

inm_t2m
inm_swe
inm_olr
inm_hlt
inm_mslp
inm_tp
inm_h500
inm_snow_cover
inm_u850
inm_v850
inm_ts
inm_ww
inm_t2m_clim
inm_ts_clim
inm_swe_clim
inm_tp_clim
t2m
t2m
t2m
t2m_clim
t2m_clim
sd
sd
sd
sd_clim
sd_clim
tp
tp
tp
tp_clim
tp_clim
sst
sst
sst
sst_clim
sst_clim
snow_cover
snow_cover
snow_cover
lsm
lsm
z
z
sdor
sdor
